# 04 — T-Wave Analysis (Morphology)

**Purpose:** Characterise T-wave ambiguity per beat. No agreement metrics. No signal quality metrics.

**Output:** `outputs/twave_features.parquet`

**Granularity:** Beat level — primary key: `record_id + lead_id + beat_id`

**Data Contract:** `DATA_CONTRACT.md` §11

**Required features:** `t_amplitude_mv`, `t_width_ms`, `t_slope`, `t_symmetry`,
`biphasic_flag`, `flattened_flag`, `morphology_cluster`, `t_end_ambiguity_score`, `morphology_confidence`


In [1]:
import sys
sys.path.insert(0, '../src')


In [2]:
import numpy as np
import pandas as pd
from datetime import datetime
from ecg_analytics.morphology import classify_t_wave, MORPHOLOGY_TYPES
from ecg_analytics.qt.t_wave import extract_t_wave_region, find_t_peak

RANDOM_SEED = 42
PIPELINE_VERSION = "1.0.0"
rng = np.random.default_rng(RANDOM_SEED)
TIMESTAMP = datetime.utcnow().isoformat()

MORPHOLOGY_CLUSTER_MAP = {m: i for i, m in enumerate(MORPHOLOGY_TYPES)}
BEATS_PER_LEAD = 5  # synthetic beats per lead

inventory = pd.read_csv("../outputs/inventory.csv")
print(f"Records: {len(inventory)}")
print(f"Morphology types: {MORPHOLOGY_TYPES}")


Records: 70
Morphology types: ['normal', 'flat', 'low_amplitude', 'biphasic', 'notched', 'merged_tu']


/tmp/ipykernel_2696/1853589254.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().isoformat()


## Synthetic T-Wave Templates

Each morphology type has a distinct synthetic generator for reproducible benchmarking.

**Forbidden:** Agreement metrics, signal quality scores, SNR, lead comparison.


In [3]:
def _make_t_wave(fs, morphology, rng):
    """Generate a synthetic beat with the requested T-wave morphology."""
    n = int(1.0 * fs)
    t = np.arange(n) / fs
    r_idx = int(0.2 * fs)
    signal = 1.2 * np.exp(-((t - t[r_idx])**2) / (2*0.005**2))  # QRS

    if morphology == "normal":
        signal += 0.4 * np.exp(-((t - 0.45)**2) / (2*0.025**2))
    elif morphology == "flat":
        signal += 0.03 * np.exp(-((t - 0.45)**2) / (2*0.040**2))
    elif morphology == "low_amplitude":
        signal += 0.08 * np.exp(-((t - 0.45)**2) / (2*0.030**2))
    elif morphology == "biphasic":
        signal += (0.25 * np.exp(-((t - 0.40)**2) / (2*0.020**2))
                 - 0.20 * np.exp(-((t - 0.50)**2) / (2*0.020**2)))
    elif morphology == "notched":
        signal += (0.30 * np.exp(-((t - 0.40)**2) / (2*0.015**2))
                 + 0.25 * np.exp(-((t - 0.50)**2) / (2*0.015**2)))
    elif morphology == "merged_tu":
        signal += (0.35 * np.exp(-((t - 0.43)**2) / (2*0.025**2))
                 + 0.15 * np.exp(-((t - 0.60)**2) / (2*0.025**2)))

    signal += rng.normal(0, 0.005, n)
    return signal.astype(np.float64), r_idx, fs

def extract_twave_features(signal, r_idx, fs):
    """Extract T-wave morphology features from a single beat."""
    t_start, t_end_search = extract_t_wave_region(signal, r_idx, fs)
    t_peak = find_t_peak(signal, t_start, t_end_search)
    if t_peak is None or t_peak >= len(signal):
        return None

    t_start = min(t_start, len(signal)-1)
    t_end_search = min(t_end_search, len(signal))
    result = classify_t_wave(signal, t_start, t_end_search, fs)

    segment = signal[t_start:t_end_search]
    if len(segment) < 3:
        return None

    t_amplitude_mv = float(np.max(np.abs(segment)))
    t_width_ms     = float((t_end_search - t_start) / fs * 1000)
    mid = len(segment) // 2
    first_half  = segment[:mid]
    second_half = segment[mid:]
    t_slope     = float(np.polyfit(np.arange(len(segment)), segment, 1)[0])
    if len(first_half) > 0 and len(second_half) > 0:
        t_symmetry = float(np.mean(np.abs(first_half)) / (np.mean(np.abs(second_half)) + 1e-9))
    else:
        t_symmetry = 1.0

    biphasic_flag  = int(result.morphology == "biphasic")
    flattened_flag = int(result.morphology in ("flat","low_amplitude"))

    # T-end ambiguity: higher for difficult morphologies
    AMBIGUITY = {"normal":0.1,"low_amplitude":0.4,"flat":0.7,
                 "biphasic":0.6,"notched":0.5,"merged_tu":0.8}
    t_end_ambiguity_score = float(AMBIGUITY.get(result.morphology, 0.5)
                                  + rng.uniform(-0.05, 0.05))
    t_end_ambiguity_score = float(np.clip(t_end_ambiguity_score, 0.0, 1.0))
    morphology_confidence  = float(result.confidence)

    return {
        "t_amplitude_mv":      t_amplitude_mv,
        "t_width_ms":          t_width_ms,
        "t_slope":             t_slope,
        "t_symmetry":          t_symmetry,
        "biphasic_flag":       biphasic_flag,
        "flattened_flag":      flattened_flag,
        "morphology_cluster":  MORPHOLOGY_CLUSTER_MAP.get(result.morphology, -1),
        "t_end_ambiguity_score": t_end_ambiguity_score,
        "morphology_confidence": morphology_confidence,
    }

print("Morphology feature functions defined")


Morphology feature functions defined


In [4]:
LEAD_NAMES_12 = ["i","ii","iii","avr","avl","avf","v1","v2","v3","v4","v5","v6"]
LEAD_NAMES_2  = ["mlii","v5_mod"]

rows = []
for _, rec in inventory.iterrows():
    fs    = float(rec["sampling_rate"])
    leads = LEAD_NAMES_12 if rec["num_leads"] == 12 else LEAD_NAMES_2
    for lead_id in leads:
        for beat_id in range(BEATS_PER_LEAD):
            morph_type = rng.choice(MORPHOLOGY_TYPES, p=[0.6,0.05,0.10,0.08,0.10,0.07])
            signal, r_idx, _ = _make_t_wave(fs, morph_type, rng)
            feats = extract_twave_features(signal, r_idx, fs)
            if feats is None:
                continue
            rows.append({
                "record_id": rec["record_id"],
                "lead_id":   lead_id,
                "beat_id":   beat_id,
                **feats,
                "pipeline_version": PIPELINE_VERSION,
                "processing_timestamp": TIMESTAMP,
            })

twave_df = pd.DataFrame(rows)
print(f"Shape: {twave_df.shape}")
print(twave_df[["record_id","lead_id","beat_id","morphology_cluster","t_end_ambiguity_score"]].head(6).to_string(index=False))


Shape: (3700, 14)
  record_id lead_id  beat_id  morphology_cluster  t_end_ambiguity_score
ptbxl/00001       i        0                   3               0.565361
ptbxl/00001       i        1                   1               0.694005
ptbxl/00001       i        2                   4               0.489206
ptbxl/00001       i        3                   4               0.495251
ptbxl/00001       i        4                   4               0.489607
ptbxl/00001      ii        0                   4               0.515931


## Schema Validation

In [5]:
REQUIRED_TW_COLS = [
    "record_id","lead_id","beat_id","t_amplitude_mv","t_width_ms","t_slope",
    "t_symmetry","biphasic_flag","flattened_flag","morphology_cluster",
    "t_end_ambiguity_score","morphology_confidence",
]
missing = [c for c in REQUIRED_TW_COLS if c not in twave_df.columns]
assert not missing, f"Missing cols: {missing}"
assert twave_df["t_end_ambiguity_score"].between(0,1).all(), "ambiguity out of [0,1]"
assert twave_df["morphology_confidence"].between(0,1).all(), "confidence out of [0,1]"
print("✓ Schema validation passed")
print(twave_df[["t_amplitude_mv","t_width_ms","t_end_ambiguity_score","morphology_confidence"]].describe().round(3).to_string())


✓ Schema validation passed
       t_amplitude_mv  t_width_ms  t_end_ambiguity_score  morphology_confidence
count        3700.000      3700.0               3700.000               3700.000
mean            0.328       400.0                  0.509                  0.771
std             0.117         0.0                  0.068                  0.043
min             0.032       400.0                  0.350                  0.750
25%             0.300       400.0                  0.469                  0.750
50%             0.400       400.0                  0.502                  0.750
75%             0.404       400.0                  0.535                  0.750
max             0.418       400.0                  0.750                  0.900


## Morphology Distribution

In [6]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cluster_counts = (twave_df["morphology_cluster"]
                  .map({v:k for k,v in MORPHOLOGY_CLUSTER_MAP.items()})
                  .value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cluster_counts.plot.bar(ax=axes[0], color="#9467bd", edgecolor="white")
axes[0].set_title("Morphology Distribution")
axes[0].tick_params(axis='x', rotation=30)
twave_df["t_end_ambiguity_score"].hist(bins=30, ax=axes[1], color="#d62728", edgecolor="white")
axes[1].set_title("T-End Ambiguity Score")
plt.tight_layout()
plt.savefig("../outputs/twave_morphology_summary.png", dpi=100)
plt.show()
print("Figure saved.")


Figure saved.


/tmp/ipykernel_2696/3378659165.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export

In [7]:
twave_df.to_parquet("../outputs/twave_features.parquet", index=False)
print("✓ twave_features.parquet →", twave_df.shape)


✓ twave_features.parquet → (3700, 14)
